# Snake Species Metadata Enrichment

Build metadata for the 109 final classification classes.

Sources:
- SnakeCLEF processed classes
- (Vietnam Snakes)[https://vietnamsnakes.com/] for Vietnamese names and available family metadata

## 1. Imports & Config

In [1]:
import pandas as pd
import requests
from bs4 import BeautifulSoup

## 2. Load Final Classes

In [2]:
classes_df = pd.read_csv("../data/processed/classes.csv")

print("Number of classes:", len(classes_df))
classes_df.head()

Number of classes: 109


,label_idx,class_id,binomial_name,MIVS,image_count
0,0,9,Achalinus rufescens,0,91
1,1,14,Acrochordus granulatus,0,152
2,2,40,Ahaetulla fusca,0,65
3,3,47,Ahaetulla prasina,0,565
4,4,66,Amphiesma stolatum,0,368


In [3]:
species_metadata = classes_df[["binomial_name", "MIVS"]].copy()
species_metadata["genus"] = (species_metadata["binomial_name"].str.split().str[0])

species_metadata["family"] = None
species_metadata["vietnamese_name"] = None

## 3. Scrape Vietnam Snakes Metadata

In [4]:
url = "https://vietnamsnakes.com/all-of-species"

response = requests.get(url, timeout=30)
response.raise_for_status()
soup = BeautifulSoup(response.text, "html.parser")

In [5]:
records = []
current_family = None

for element in soup.find_all(["div", "a"]):
    # Detect family headings
    if element.name == "div":
        for text in element.stripped_strings:
            if text.endswith("idae") or "idae -" in text:
                current_family = text.split(" - ")[0].strip()

    # Detect species cards
    if element.name == "a":
        binomial_tag = element.find("i")

        if binomial_tag is None:
            continue

        binomial_name = binomial_tag.get_text(strip=True)

        # Scientific names should contain at least genus + species
        if len(binomial_name.split()) < 2:
            continue

        vietnamese_tag = element.select_one(".fw-600.fs-7.text-truncate.text-truncate-1")

        if vietnamese_tag is None:
            continue

        vietnamese_name = vietnamese_tag.get_text(" ", strip=True)

        records.append(
            {
                "binomial_name": binomial_name,
                "vietnamese_name": vietnamese_name,
                "family": current_family,
            }
        )

vn_snakes = (pd.DataFrame(records).drop_duplicates(subset="binomial_name").reset_index(drop=True))
vn_snakes.head()

,binomial_name,vietnamese_name,family
0,Acrochordus granulatus,"Rắn rầm ri cá, Đẻn cườm",Acrochordidae
1,Acrochordus javanicus,"Rắn rầm ri cóc, Rầm ri hạt",Acrochordidae
2,Calamaria abramovi,Rắn mai gầm A-b-a-mo,Calamariidae
3,Calamaria buchi,Rắn mai gầm Búc,Calamariidae
4,Calamaria cf. pavimentata,Rắn mai gầm lát,Calamariidae


In [6]:
# fix name
vn_snakes["match_name"] = (vn_snakes["binomial_name"].str.replace(r"\bcf\.\s*", "", regex=True) .str.replace(r"\s+", " ", regex=True).str.strip())
vn_snakes.head()

,binomial_name,vietnamese_name,family,match_name
0,Acrochordus granulatus,"Rắn rầm ri cá, Đẻn cườm",Acrochordidae,Acrochordus granulatus
1,Acrochordus javanicus,"Rắn rầm ri cóc, Rầm ri hạt",Acrochordidae,Acrochordus javanicus
2,Calamaria abramovi,Rắn mai gầm A-b-a-mo,Calamariidae,Calamaria abramovi
3,Calamaria buchi,Rắn mai gầm Búc,Calamariidae,Calamaria buchi
4,Calamaria cf. pavimentata,Rắn mai gầm lát,Calamariidae,Calamaria pavimentata


## 4. Mapping and fill in `species_metadata`

In [7]:
name_map = (vn_snakes.drop_duplicates("match_name").set_index("match_name")["vietnamese_name"])
family_map = (vn_snakes.drop_duplicates("match_name").set_index("match_name")["family"])
species_metadata["vietnamese_name"] = (species_metadata["binomial_name"].map(name_map))
species_metadata["family"] = (species_metadata["binomial_name"].map(family_map))

In [8]:
print("Missing Vietnamese name:", species_metadata["vietnamese_name"].isna().sum())
print("Missing family:", species_metadata["family"].isna().sum())

Missing Vietnamese name: 24
Missing family: 24


## 5. Complete Missing Vietnamese Names

Some species could not be matched automatically because of taxonomy or naming differences.
Verified Vietnamese names, family are added manually for these cases.

In [9]:
manual_name_map = {
    "Achalinus rufescens": "Rắn xe điếu nâu",
    "Boiga dendrophila": "Rắn rào khoang vàng",
    "Boiga drapiezii": "Rắn mèo đốm trắng",
    "Bungarus fasciatus": "Rắn cạp nong",
    "Bungarus flaviceps": "Rắn cạp nong đầu đỏ",
    "Bungarus multicinctus": "Rắn cạp nia bắc",
    "Calliophis intestinalis": "Rắn lá khô sọc",
    "Cylindrophis ruffus": "Rắn trun",
    "Emydocephalus annulatus": "Đẻn đầu rùa",
    "Fowlea piscator": "Rắn nước",
    "Homalopsis buccata": "Rắn ri cá",
    "Hydrophis curtus": "Đẻn cơm",
    "Hydrophis ornatus": "Rắn đẻn đuôi sọc",
    "Hydrophis platurus": "Đẻn đuôi vàng, Đẻn sọc dưa",
    "Hydrophis schistosus": "Rắn đẻn mỏ",
    "Pareas carinatus": "Rắn hổ mây gờ",
    "Rhabdophis tigrinus": "Rắn cỏ Nhật",
    "Sibynophis melanocephalus": "Rắn rồng đầu đen",
    "Sinomicrurus macclellandi": "Rắn san hô MacClelland, Rắn san hô đầu bạc",
    "Trimeresurus gumprechti": "Rắn lục xanh Gumprecht",
    "Trimeresurus guoi": "Rắn lục Guo",
    "Trimeresurus macrops": "Rắn lục xanh mắt to",
    "Tropidolaemus wagleri": "Rắn lục hoa cân",
    "Xenochrophis trianguligerus": "Rắn nước vân tam giác",
}

manual_family_map = {
    "Achalinus rufescens": "Xenodermidae",
    "Boiga dendrophila": "Colubridae",
    "Boiga drapiezii": "Colubridae",
    "Bungarus fasciatus": "Elapidae",
    "Bungarus flaviceps": "Elapidae",
    "Bungarus multicinctus": "Elapidae",
    "Calliophis intestinalis": "Elapidae",
    "Cylindrophis ruffus": "Cylindrophiidae",
    "Emydocephalus annulatus": "Elapidae",
    "Fowlea piscator": "Colubridae",
    "Homalopsis buccata": "Homalopsidae",
    "Hydrophis curtus": "Elapidae",
    "Hydrophis ornatus": "Elapidae",
    "Hydrophis platurus": "Elapidae",
    "Hydrophis schistosus": "Elapidae",
    "Pareas carinatus": "Pareidae",
    "Rhabdophis tigrinus": "Colubridae",
    "Sibynophis melanocephalus": "Colubridae",
    "Sinomicrurus macclellandi": "Elapidae",
    "Trimeresurus gumprechti": "Viperidae",
    "Trimeresurus guoi": "Viperidae",
    "Trimeresurus macrops": "Viperidae",
    "Tropidolaemus wagleri": "Viperidae",
    "Xenochrophis trianguligerus": "Colubridae",
}

In [10]:
missing_name_mask = species_metadata["vietnamese_name"].isna()
missing_family_mask = species_metadata["family"].isna()

species_metadata.loc[missing_name_mask, "vietnamese_name"] = (species_metadata.loc[missing_name_mask, "binomial_name"].map(manual_name_map))
species_metadata.loc[missing_family_mask, "family"] = (species_metadata.loc[missing_family_mask, "binomial_name"].map(manual_family_map))

In [11]:
print("Missing Vietnamese name:", species_metadata["vietnamese_name"].isna().sum())
print("Missing family:",species_metadata["family"].isna().sum())

Missing Vietnamese name: 0
Missing family: 0


## 6. Final checking

In [12]:
print("Rows:", len(species_metadata))
print("Duplicate binomial_name:", species_metadata["binomial_name"].duplicated().sum())
print("Missing binomial_name:", species_metadata["binomial_name"].isna().sum())
print("Missing Vietnamese name:", species_metadata["vietnamese_name"].isna().sum())
print("Missing family:", species_metadata["family"].isna().sum())
print("Missing genus:", species_metadata["genus"].isna().sum())
print("Missing MIVS:", species_metadata["MIVS"].isna().sum())

Rows: 109
Duplicate binomial_name: 0
Missing binomial_name: 0
Missing Vietnamese name: 0
Missing family: 0
Missing genus: 0
Missing MIVS: 0


## 7. Export Result

In [13]:
species_metadata = species_metadata[
    [
        "binomial_name",
        "vietnamese_name",
        "family",
        "genus",
        "MIVS",
    ]
]

species_metadata.to_csv("../data/processed/species_metadata.csv", index=False,)
species_metadata.head()

,binomial_name,vietnamese_name,family,genus,MIVS
0,Achalinus rufescens,Rắn xe điếu nâu,Xenodermidae,Achalinus,0
1,Acrochordus granulatus,"Rắn rầm ri cá, Đẻn cườm",Acrochordidae,Acrochordus,0
2,Ahaetulla fusca,"Rắn roi mõm nhọn, Rắn lục kim",Colubridae,Ahaetulla,0
3,Ahaetulla prasina,"Rắn roi thường, Rắn lục kim",Colubridae,Ahaetulla,0
4,Amphiesma stolatum,"Rắn sãi cỏ, Rắn sãi thường",Colubridae,Amphiesma,0


In [14]:
pd.read_csv("../data/processed/species_metadata.csv").shape

(109, 5)